# Exercício 20 — mercantil: relatórios em **paralelo** (LCEL + Pydantic) e **agente** de análise

**Objectivos**

- Simular **compras** (entrada de mercadoria) e **vendas** num mercantil com catálogo fixo.
- Modelar dados com **Pydantic v2** (`NotaCompra`, `NotaVenda`, `RelatorioEstoque`, `RelatorioLucroVendas`, …).
- Orquestrar dois relatórios independentes com **`RunnableParallel`**: o mesmo `MercantilSnapshot` alimenta os dois ramos.
- Usar um **agente ReAct** (LangGraph + Gemini) com *tools* que devolvem os JSON dos relatórios, para uma **análise integrada** de stock e de vendas/lucro.

**Ficheiros:** `mercantil_paralelo.py` (simulação + LCEL), `mercantil_agente_analise.py` (agente).

**Arranque:** `./run.sh` nesta pasta. **Gemini** (`GOOGLE_API_KEY`, `GEMINI_MODEL_EX20`) necessário para o agente e para a secção opcional de sínteses paralelas.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

here = Path.cwd().resolve()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

from mercantil_paralelo import (
    LinhaCompra,
    LinhaVenda,
    Mercantil,
    NotaCompra,
    NotaVenda,
    ProdutoCatalogo,
    SaidaRelatoriosParalelos,
    pipeline_relatorios_paralelos,
)

loja = Mercantil()
for p in (
    ProdutoCatalogo(sku="ARROZ1", nome="Arroz 1 kg", preco_compra_unitario="1.20", preco_venda_unitario="1.89"),
    ProdutoCatalogo(sku="OLEO1", nome="Óleo 900 ml", preco_compra_unitario="2.50", preco_venda_unitario="3.49"),
    ProdutoCatalogo(sku="ACUC1", nome="Açúcar 1 kg", preco_compra_unitario="0.95", preco_venda_unitario="1.29"),
):
    loja.registar_produto(p)

loja.aplicar_compra(
    NotaCompra(
        referencia="C-2026-001",
        linhas=[
            LinhaCompra(sku="ARROZ1", quantidade=200, preco_unitario="1.18"),
            LinhaCompra(sku="OLEO1", quantidade=80, preco_unitario="2.45"),
            LinhaCompra(sku="ACUC1", quantidade=120, preco_unitario="0.92"),
        ],
    )
)

loja.aplicar_venda(
    NotaVenda(
        referencia="V-2026-010",
        linhas=[
            LinhaVenda(sku="ARROZ1", quantidade=40),
            LinhaVenda(sku="OLEO1", quantidade=15),
        ],
    )
)
loja.aplicar_venda(
    NotaVenda(
        referencia="V-2026-011",
        linhas=[LinhaVenda(sku="ACUC1", quantidade=30, preco_unitario_venda="1.35")],
    )
)

chain = pipeline_relatorios_paralelos()
saida = chain.invoke(loja)

print("Chaves devolvidas pelo RunnableParallel:", list(saida.keys()))
topo = SaidaRelatoriosParalelos.model_validate(saida)
print("\n=== Relatório de estoque (Pydantic) ===")
print(topo.estoque.model_dump_json(indent=2))
print("\n=== Relatório de lucro e vendas (Pydantic) ===")
print(topo.lucro_vendas.model_dump_json(indent=2))
print("\nLucro bruto:", topo.lucro_vendas.lucro_bruto)


Chaves devolvidas pelo RunnableParallel: ['estoque', 'lucro_vendas']

=== Relatório de estoque (Pydantic) ===
{
  "emitido_em": "2026-05-14T22:51:45.888127Z",
  "linhas": [
    {
      "sku": "ACUC1",
      "nome_produto": "Açúcar 1 kg",
      "quantidade_em_stock": 90,
      "preco_compra_referencia": "0.95",
      "valor_em_stock": "85.50"
    },
    {
      "sku": "ARROZ1",
      "nome_produto": "Arroz 1 kg",
      "quantidade_em_stock": 160,
      "preco_compra_referencia": "1.20",
      "valor_em_stock": "192.00"
    },
    {
      "sku": "OLEO1",
      "nome_produto": "Óleo 900 ml",
      "quantidade_em_stock": 65,
      "preco_compra_referencia": "2.50",
      "valor_em_stock": "162.50"
    }
  ],
  "valor_total_em_stock": "440.00"
}

=== Relatório de lucro e vendas (Pydantic) ===
{
  "emitido_em": "2026-05-14T22:51:45.888127Z",
  "detalhe_por_linha": [
    {
      "referencia_nota": "V-2026-010",
      "sku": "ARROZ1",
      "nome_produto": "Arroz 1 kg",
      "quantidade": 40,

## Agente ReAct — análise de **stock** e de **vendas / lucro**

O grafo em `mercantil_agente_analise.py` expõe três *tools* alimentadas pelos mesmos modelos Pydantic (`relatorio_estoque`, `relatorio_lucro_vendas`). O modelo **decide** quando chamar cada uma e produz a análise final em texto (secções pedidas no *system prompt*).

Requer **`GOOGLE_API_KEY`** (ou `GEMINI_API_KEY`) no `.env` da raiz.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env)
        break

from mercantil_agente_analise import (
    config_analise,
    criar_grafo_analise_mercantil,
    ultima_resposta_texto,
)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    print("Defina GOOGLE_API_KEY ou GEMINI_API_KEY no `.env` da raiz para correr o agente.")
else:
    grafico = criar_grafo_analise_mercantil(loja)
    cfg = config_analise("thread-ex20-analise-1")
    out = grafico.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "Elabora uma análise completa de **stock** e de **vendas e lucro** desta loja. "
                        "Usa as ferramentas e segue a estrutura pedida no sistema."
                    )
                )
            ]
        },
        cfg,
    )
    texto = ultima_resposta_texto(out)
    print(texto if texto else out)

Análise de Stock e Vendas & Lucro

1.  **Stock**

*   **Cobertura por SKU:**
    *   Açúcar 1 kg (ACUC1): 90 unidades
    *   Arroz 1 kg (ARROZ1): 160 unidades
    *   Óleo 900 ml (OLEO1): 65 unidades
*   **Valor imobilizado:** O valor total em stock é de 440.00.
    *   Açúcar 1 kg (ACUC1): 85.50
    *   Arroz 1 kg (ARROZ1): 192.00
    *   Óleo 900 ml (OLEO1): 162.50
*   **Risco de rutura ou excesso:** Não há indicação direta de risco de rutura ou excesso, mas a análise de vendas (secção 2) pode ajudar a refinar esta avaliação.

2.  **Vendas**

*   **Mix por produto:**
    *   Arroz 1 kg (ARROZ1): 75.60 de receita (40 unidades vendidas)
    *   Óleo 900 ml (OLEO1): 52.35 de receita (15 unidades vendidas)
    *   Açúcar 1 kg (ACUC1): 40.50 de receita (30 unidades vendidas)
*   **Preços efectivos vs. catálogo:** Não há informação de preços de catálogo, impossibilitando a comparação. Os preços efectivos foram:
    *   Arroz 1 kg (ARROZ1): 1.89
    *   Óleo 900 ml (OLEO1): 3.49
    *   Aç

## Opcional — duas **sínteses** em paralelo com Gemini (`with_structured_output`)

Cada ramo recebe o JSON de um relatório e devolve um modelo Pydantic curto. Requer **`GOOGLE_API_KEY`** no `.env` da raiz.

In [3]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env)
        break

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    print("Sem GOOGLE_API_KEY / GEMINI_API_KEY — célula ignorada.")
else:
    modelo = (os.environ.get("GEMINI_MODEL_EX20") or os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").strip()
    os.environ.setdefault("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", ""))

    class SinteseEstoque(BaseModel):
        bullets: list[str] = Field(description="3 a 5 bullets em português europeu sobre risco de ruptura e valor imobilizado")

    class SinteseFinanceira(BaseModel):
        bullets: list[str] = Field(description="3 a 5 bullets sobre receita, CMV e margem")

    llm = ChatGoogleGenerativeAI(model=modelo, temperature=0.2)
    p_est = ChatPromptTemplate.from_messages(
        [("human", "Resume para o gestor o relatório de stock (JSON):\n{json}")]
    )
    p_fin = ChatPromptTemplate.from_messages(
        [("human", "Resume para o gestor o relatório financeiro (JSON):\n{json}")]
    )

    c_est = p_est | llm.with_structured_output(SinteseEstoque)
    c_fin = p_fin | llm.with_structured_output(SinteseFinanceira)

    sinteses = RunnableParallel(
        sintese_estoque=RunnableLambda(lambda d: {"json": d["estoque"].model_dump_json()}) | c_est,
        sintese_financeira=RunnableLambda(lambda d: {"json": d["lucro_vendas"].model_dump_json()}) | c_fin,
    )

    doc = sinteses.invoke({"estoque": topo.estoque, "lucro_vendas": topo.lucro_vendas})
    print(json.dumps({k: v.model_dump() for k, v in doc.items()}, ensure_ascii=False, indent=2))


{
  "sintese_estoque": {
    "bullets": [
      "O relatório de stock, emitido em 2026-05-14, indica um valor total em stock de 440.00.",
      "O produto com maior quantidade em stock é o Arroz 1 kg, com 160 unidades, representando um valor de 192.00.",
      "O Açúcar 1 kg apresenta 90 unidades em stock, com um valor de 85.50.",
      "O Óleo 900 ml tem 65 unidades em stock, representando um valor de 162.50.",
      "É importante monitorizar o stock de Óleo 900 ml para evitar ruturas, dado que a quantidade é relativamente baixa comparada com os outros produtos."
    ]
  },
  "sintese_financeira": {
    "bullets": [
      "Receita total: R$168.45",
      "Custo total da mercadoria vendida: R$114.00",
      "Lucro bruto: R$54.45",
      "Destaque para a venda de Arroz 1 kg (SKU: ARROZ1) com maior receita (R$75.60) e margem (R$27.60).",
      "Óleo 900 ml (SKU: OLEO1) apresentou a maior margem unitária, apesar da menor quantidade vendida."
    ]
  }
}
